# Phase 7: SchNet Optuna on 300k molecules (Kaggle GPU)

## Setup
1. Upload `pyg_3d_graphs_etkdg_300k.pt` to Kaggle dataset (e.g. `molgap-300k`)
2. Enable GPU accelerator (T4 x2)
3. Run all cells

## Safety
- Every epoch saves `checkpoint.pt` + `best_model.pt` to Kaggle output
- Optional: mount Google Drive or push to HuggingFace Hub for redundancy
- Progressive scaling: profile on 10k first, then scale up

In [ ]:
!pip install -q torch-geometric optuna huggingface_hub
!pip install -q pyg-lib -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__.split('+')[0])")+cu$(python -c "import torch; print(torch.version.cuda.replace('.',''))").html

In [ ]:
import os, time, json, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

# Performance: enable cudnn autotuner
torch.backends.cudnn.benchmark = True

print(f"torch {torch.__version__}, CUDA {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}, {props.total_mem / 1e9:.1f} GB")
print(f"Using GPU 0 (PyG does not support DataParallel)")

In [ ]:
# ══════════════════════════════════════════════════════════
# CONFIG — adjust these before running
# ══════════════════════════════════════════════════════════
GRAPH_PATH = "/kaggle/input/molgap-300k/pyg_3d_graphs_etkdg_300k.pt"
OUTPUT_DIR = "/kaggle/working"

# Progressive scaling: set DATA_LIMIT to test smaller subsets first
# None = use all data, or set to 10000 / 50000 / 100000 for profiling
DATA_LIMIT = None

# Google Drive backup (optional, set to None to skip)
GDRIVE_DIR = None  # e.g. "/content/drive/MyDrive/molgap/phase7"

# HuggingFace Hub backup (optional, set to None to skip)
HF_REPO = None  # e.g. "your-username/molgap-phase7"
HF_TOKEN = None  # or set env var HF_TOKEN

# DataLoader performance (Kaggle has 4 CPU cores)
NUM_WORKERS = 2
PIN_MEMORY = True

SEED = 42
TARGET_COLS = ["homo", "lumo", "gap"]
N_TRIALS = 20
FAST_EPOCHS = 80
FULL_EPOCHS = 500
FULL_PATIENCE = 40
CHECKPOINT_EVERY = 1  # save checkpoint every N epochs

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ── Checkpoint helpers ──
def save_checkpoint(state, filename, backup_dirs=None):
    """Save checkpoint to OUTPUT_DIR and optionally to backup locations."""
    path = os.path.join(OUTPUT_DIR, filename)
    torch.save(state, path)
    # Backup to Google Drive
    if GDRIVE_DIR:
        os.makedirs(GDRIVE_DIR, exist_ok=True)
        shutil.copy2(path, os.path.join(GDRIVE_DIR, filename))


def push_to_hf(filename):
    """Push a file to HuggingFace Hub."""
    if not HF_REPO:
        return
    try:
        from huggingface_hub import HfApi
        token = HF_TOKEN or os.environ.get("HF_TOKEN")
        api = HfApi()
        api.upload_file(
            path_or_fileobj=os.path.join(OUTPUT_DIR, filename),
            path_in_repo=filename,
            repo_id=HF_REPO,
            repo_type="model",
            token=token,
        )
        print(f"    -> pushed {filename} to HF:{HF_REPO}")
    except Exception as e:
        print(f"    HF push failed: {e}")


def save_and_backup(state, filename):
    save_checkpoint(state, filename)
    push_to_hf(filename)

In [ ]:
# ── SchNet model ──
class SchNetWrapper(nn.Module):
    def __init__(self, hidden_channels, num_filters, num_interactions,
                 num_gaussians, cutoff, dropout=0.1, n_targets=3,
                 use_charges=False):
        super().__init__()
        from torch_geometric.nn.models import SchNet
        self.schnet = SchNet(
            hidden_channels=hidden_channels, num_filters=num_filters,
            num_interactions=num_interactions, num_gaussians=num_gaussians,
            cutoff=cutoff,
        )
        self.use_charges = use_charges
        if use_charges:
            self.charge_proj = nn.Linear(1, hidden_channels)
        self.head = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.SiLU(),
            nn.Linear(hidden_channels // 2, n_targets),
        )

    def forward(self, z, pos, batch, charges=None):
        from torch_geometric.nn import global_mean_pool
        from torch_geometric.nn.models.schnet import radius_graph
        h = self.schnet.embedding(z)
        if self.use_charges and charges is not None:
            h = h + self.charge_proj(charges.unsqueeze(-1))
        edge_index = radius_graph(pos, r=self.schnet.cutoff, batch=batch, max_num_neighbors=32)
        row, col = edge_index
        edge_weight = (pos[row] - pos[col]).norm(dim=-1)
        edge_attr = self.schnet.distance_expansion(edge_weight)
        for interaction in self.schnet.interactions:
            h = h + interaction(h, edge_index, edge_weight, edge_attr)
        h = global_mean_pool(h, batch)
        return self.head(h)

In [ ]:
# ── Load data ──
print("Loading graphs...")
data_list = torch.load(GRAPH_PATH, weights_only=False)
print(f"Loaded {len(data_list)} graphs")

# Apply DATA_LIMIT for progressive scaling
if DATA_LIMIT and DATA_LIMIT < len(data_list):
    np.random.seed(SEED)
    subset_idx = np.random.choice(len(data_list), DATA_LIMIT, replace=False)
    data_list = [data_list[i] for i in sorted(subset_idx)]
    print(f"  -> Subset to {len(data_list)} for profiling")

has_charges = hasattr(data_list[0], 'charges')
print(f"Gasteiger charges: {has_charges}")

# train/valid/test split
all_idx = np.arange(len(data_list))
train_valid_idx, test_idx = train_test_split(all_idx, test_size=0.1, random_state=SEED)
train_idx, valid_idx = train_test_split(train_valid_idx, test_size=0.1/0.9, random_state=SEED)

train_data = [data_list[i] for i in train_idx]
valid_data = [data_list[i] for i in valid_idx]
test_data  = [data_list[i] for i in test_idx]

# normalization
train_y = np.stack([d.y.squeeze(0).numpy() for d in train_data])
y_mean = train_y.mean(axis=0)
y_std = train_y.std(axis=0)
y_std[y_std < 1e-6] = 1.0

for d in train_data + valid_data + test_data:
    d.y = (d.y - torch.tensor(y_mean)) / torch.tensor(y_std)

del data_list
n_total = len(train_data) + len(valid_data) + len(test_data)
print(f"Split: train={len(train_data)}, valid={len(valid_data)}, test={len(test_data)}")
print(f"y_mean={y_mean}, y_std={y_std}")

In [ ]:
# ── Training with per-epoch checkpoint ──
def train_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss, n = 0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            charges = getattr(batch, 'charges', None)
            out = model(batch.z, batch.pos, batch.batch, charges=charges)
            loss = F.l1_loss(out, batch.y)
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * batch.num_graphs
        n += batch.num_graphs
    return total_loss / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    for batch in loader:
        batch = batch.to(device)
        with torch.amp.autocast("cuda"):
            charges = getattr(batch, 'charges', None)
            out = model(batch.z, batch.pos, batch.batch, charges=charges)
        preds.append(out.cpu().numpy())
        trues.append(batch.y.cpu().numpy())
    return np.vstack(preds), np.vstack(trues)


def run_training(params, train_loader, valid_loader,
                 max_epochs, patience, verbose=False,
                 save_ckpt=False, phase_label=""):
    """Train SchNet with per-epoch checkpointing."""
    model = SchNetWrapper(
        hidden_channels=params["hidden_channels"],
        num_filters=params["num_filters"],
        num_interactions=params["num_interactions"],
        num_gaussians=params["num_gaussians"],
        cutoff=params["cutoff"],
        dropout=params["dropout"],
        use_charges=has_charges,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=params["lr"], weight_decay=params["weight_decay"])
    if params["scheduler"] == "cosine":
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)
    else:
        scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5,
                                      patience=max(5, patience // 3), min_lr=1e-6)
    scaler = torch.amp.GradScaler("cuda")

    best_val_mae = float("inf")
    best_epoch = 0
    best_state = None
    log_rows = []

    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, scaler)

        if not np.isfinite(train_loss):
            if verbose:
                print(f"  NaN detected at epoch {epoch}, stopping.")
            break

        val_pred, val_true = evaluate(model, valid_loader)
        val_pred_real = val_pred * y_std + y_mean
        val_true_real = val_true * y_std + y_mean
        val_mae = float(np.mean(np.abs(val_pred_real - val_true_real)))

        if params["scheduler"] == "cosine":
            scheduler.step(epoch)
        else:
            scheduler.step(val_mae)

        elapsed = time.time() - t0
        log_rows.append({"epoch": epoch, "train_loss": train_loss,
                         "val_mae": val_mae, "lr": optimizer.param_groups[0]["lr"],
                         "time_s": elapsed})

        is_best = val_mae < best_val_mae
        if is_best:
            best_val_mae = val_mae
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if verbose and (epoch % 10 == 0 or epoch == 1 or is_best):
            marker = " *" if is_best else ""
            print(f"  Epoch {epoch:3d} | loss={train_loss:.4f} | val_MAE={val_mae:.4f} | "
                  f"best={best_val_mae:.4f}@{best_epoch} | "
                  f"lr={optimizer.param_groups[0]['lr']:.1e} | {elapsed:.1f}s{marker}")

        if save_ckpt and epoch % CHECKPOINT_EVERY == 0:
            ckpt = {
                "epoch": epoch,
                "model_state_dict": {k: v.cpu().clone() for k, v in model.state_dict().items()},
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "best_val_mae": best_val_mae,
                "best_epoch": best_epoch,
                "params": params,
                "y_mean": y_mean.tolist(),
                "y_std": y_std.tolist(),
                "log_rows": log_rows,
            }
            save_checkpoint(ckpt, "checkpoint.pt")
            if is_best:
                save_and_backup(best_state, "best_model.pt")

        if epoch - best_epoch >= patience:
            if verbose:
                print(f"  Early stop at epoch {epoch} (best={best_epoch})")
            break

    if save_ckpt and best_state is not None:
        save_and_backup(best_state, "best_model.pt")
        pd.DataFrame(log_rows).to_csv(f"{OUTPUT_DIR}/retrain_log_300k.csv", index=False)

    return best_val_mae, best_epoch, best_state, log_rows

## Step 1: Profile on small subset

Before running the full Optuna search, profile a single training run to measure:
- GPU utilization
- Epoch time
- Memory usage

Set `DATA_LIMIT = 10000` above and run this cell first.

In [ ]:
# ── Profile run: 5 epochs with default params ──
print("=" * 60)
print(f"Profile run: {n_total} molecules, 5 epochs")
print("=" * 60)

profile_params = {
    "hidden_channels": 192, "num_filters": 256, "num_interactions": 6,
    "num_gaussians": 100, "cutoff": 5.0, "dropout": 0.1,
    "lr": 5e-4, "weight_decay": 1e-4,
    "scheduler": "plateau",
}

for bs in [32, 64, 128]:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    profile_params["batch_size"] = bs
    loader_t = DataLoader(train_data, batch_size=bs, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    loader_v = DataLoader(valid_data, batch_size=bs,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    t0 = time.time()
    try:
        _, _, _, logs = run_training(
            profile_params, loader_t, loader_v,
            max_epochs=5, patience=99, verbose=False)
        avg_time = np.mean([r["time_s"] for r in logs])
        peak_mem = torch.cuda.max_memory_allocated() / 1e9
        print(f"  bs={bs:3d} | epoch={avg_time:.1f}s | peak_VRAM={peak_mem:.2f}GB | "
              f"est_full={avg_time * FULL_EPOCHS / 3600:.1f}h")
    except RuntimeError as e:
        if "out of memory" in str(e):
            print(f"  bs={bs:3d} | OOM!")
            torch.cuda.empty_cache()
        else:
            raise

print(f"\nPick the largest bs that fits, then proceed to Optuna.")

## Step 2: Optuna search

After profiling, set `DATA_LIMIT = None` (or your chosen scale) and re-run from the "Load data" cell, then run the Optuna cell below.

In [ ]:
# ── Optuna search ──
import optuna

def objective(trial):
    params = {
        "hidden_channels": trial.suggest_categorical("hidden_channels", [128, 192, 256]),
        "num_filters": trial.suggest_categorical("num_filters", [128, 192, 256]),
        "num_interactions": trial.suggest_int("num_interactions", 3, 6),
        "num_gaussians": trial.suggest_categorical("num_gaussians", [25, 50, 100]),
        "cutoff": trial.suggest_categorical("cutoff", [5.0, 6.0, 8.0, 10.0]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3, step=0.05),
        "lr": trial.suggest_float("lr", 1e-4, 1e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
        "scheduler": trial.suggest_categorical("scheduler", ["plateau", "cosine"]),
    }

    train_loader = DataLoader(train_data, batch_size=params["batch_size"], shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    valid_loader = DataLoader(valid_data, batch_size=params["batch_size"],
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    try:
        best_mae, best_ep, _, _ = run_training(
            params, train_loader, valid_loader,
            max_epochs=FAST_EPOCHS, patience=15)
    except Exception as e:
        print(f"  Trial {trial.number} failed: {e}")
        torch.cuda.empty_cache()
        return float("inf")

    print(f"  Trial {trial.number:2d} | MAE={best_mae:.4f} @ ep{best_ep} | "
          f"h={params['hidden_channels']} int={params['num_interactions']} "
          f"cut={params['cutoff']:.0f} lr={params['lr']:.1e} bs={params['batch_size']}")
    torch.cuda.empty_cache()
    return best_mae


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(objective, n_trials=N_TRIALS)

print(f"\nBest trial {study.best_trial.number}: MAE={study.best_value:.4f}")
print(f"Params: {study.best_params}")

study.trials_dataframe().to_csv(f"{OUTPUT_DIR}/optuna_trials_300k.csv", index=False)
with open(f"{OUTPUT_DIR}/optuna_best_params_300k.json", "w") as f:
    json.dump(study.best_params, f, indent=2)

## Step 3: Full retrain with best params

This saves `checkpoint.pt` every epoch and `best_model.pt` whenever validation improves.

In [ ]:
# ── Full retrain with checkpointing ──
print(f"\n{'='*60}")
print(f"Full retrain: {FULL_EPOCHS} epochs, patience={FULL_PATIENCE}")
print(f"Checkpoint every {CHECKPOINT_EVERY} epoch(s)")
print(f"{'='*60}\n")

best_params = dict(study.best_params)
train_loader = DataLoader(train_data, batch_size=best_params["batch_size"], shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
valid_loader = DataLoader(valid_data, batch_size=best_params["batch_size"],
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_data,  batch_size=best_params["batch_size"],
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

best_mae, best_epoch, best_state, log_rows = run_training(
    best_params, train_loader, valid_loader,
    max_epochs=FULL_EPOCHS, patience=FULL_PATIENCE,
    verbose=True, save_ckpt=True)

print(f"\nBest val MAE: {best_mae:.4f} @ epoch {best_epoch}")

In [ ]:
# ── Resume from checkpoint (if session restarted) ──
# Uncomment and run this cell if you need to resume after a Kaggle session restart.
# It loads the checkpoint and continues training from where it left off.

# RESUME = True
# if RESUME:
#     ckpt = torch.load(f"{OUTPUT_DIR}/checkpoint.pt", weights_only=False)
#     best_params = ckpt["params"]
#     start_epoch = ckpt["epoch"] + 1
#     y_mean = np.array(ckpt["y_mean"])
#     y_std = np.array(ckpt["y_std"])
#     print(f"Resuming from epoch {start_epoch}, best MAE={ckpt['best_val_mae']:.4f}@{ckpt['best_epoch']}")
#     print(f"Params: {best_params}")
#     
#     # Rebuild model + optimizer + scheduler from checkpoint
#     model = SchNetWrapper(
#         hidden_channels=best_params["hidden_channels"],
#         num_filters=best_params["num_filters"],
#         num_interactions=best_params["num_interactions"],
#         num_gaussians=best_params["num_gaussians"],
#         cutoff=best_params["cutoff"],
#         dropout=best_params["dropout"],
#         use_charges=has_charges,
#     ).to(device)
#     model.load_state_dict({k: v.to(device) for k, v in ckpt["model_state_dict"].items()})
#     
#     # Then continue training manually or re-run the full retrain cell
#     # (the checkpoint saves enough state to rebuild everything)

In [ ]:
# ── Evaluate on test set ──
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = SchNetWrapper(
    hidden_channels=best_params["hidden_channels"],
    num_filters=best_params["num_filters"],
    num_interactions=best_params["num_interactions"],
    num_gaussians=best_params["num_gaussians"],
    cutoff=best_params["cutoff"],
    dropout=best_params["dropout"],
    use_charges=has_charges,
).to(device)
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

test_pred, test_true = evaluate(model, test_loader)
test_pred_real = test_pred * y_std + y_mean
test_true_real = test_true * y_std + y_mean

print(f"\n{'='*60}")
print(f"  Phase 7: SchNet {n_total//1000}k Test Results")
print(f"{'='*60}")

metrics = {}
for i, t in enumerate(TARGET_COLS):
    mae = mean_absolute_error(test_true_real[:, i], test_pred_real[:, i])
    rmse = np.sqrt(mean_squared_error(test_true_real[:, i], test_pred_real[:, i]))
    r2 = r2_score(test_true_real[:, i], test_pred_real[:, i])
    metrics[t] = {"mae": float(mae), "rmse": float(rmse), "r2": float(r2)}
    print(f"  {t:5s}: MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")

avg_mae = np.mean([m["mae"] for m in metrics.values()])
avg_rmse = np.mean([m["rmse"] for m in metrics.values()])
avg_r2 = np.mean([m["r2"] for m in metrics.values()])
metrics["average"] = {"mae": float(avg_mae), "rmse": float(avg_rmse), "r2": float(avg_r2)}
print(f"  avg  : MAE={avg_mae:.4f}  RMSE={avg_rmse:.4f}  R2={avg_r2:.4f}")

print(f"\n  vs Phase 6 (44.8k):")
print(f"    Phase 6: avg MAE=0.162  R2=0.882")
print(f"    Phase 7: avg MAE={avg_mae:.3f}  R2={avg_r2:.3f}")

In [ ]:
# ── Save all outputs ──
save_and_backup(best_state, "gnn_schnet_3d_optuna_300k.pt")

final_meta = {
    "model": "SchNet_3D_ETKDG_300k",
    "params": best_params,
    "n_data": n_total,
    "mw_range": "200-1000",
    "best_epoch": best_epoch,
    "epochs_trained": len(log_rows),
    "n_trials": N_TRIALS,
    "metrics": metrics,
    "y_mean": y_mean.tolist(),
    "y_std": y_std.tolist(),
}
with open(f"{OUTPUT_DIR}/schnet_optuna_300k_metrics.json", "w") as f:
    json.dump(final_meta, f, indent=2)

# Push final results
for fname in ["schnet_optuna_300k_metrics.json", "optuna_trials_300k.csv",
              "optuna_best_params_300k.json", "retrain_log_300k.csv"]:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        if GDRIVE_DIR:
            shutil.copy2(fpath, os.path.join(GDRIVE_DIR, fname))
        push_to_hf(fname)

print(f"\nAll outputs saved to {OUTPUT_DIR}/")
print(f"Download and place locally:")
print(f"  gnn_schnet_3d_optuna_300k.pt  -> models/")
print(f"  *.json, *.csv                 -> results/phase7/")